# Identifying Phrases and Splitting Compounds

Tokenization decides what a token is; stemming decides which tokens are equivalent.
Both work one word at a time. This demo handles the two places where a one-word
view of meaning breaks down. Phrases pack several words into one concept: "New York"
is not "New" plus "York", and a query for "New" retrieves the wrong documents.
Compounds do the opposite: the German "Bücherregal" (bookcase) is one token built
from "Bücher" (books) and "Regal" (shelf), so a shopper searching for "Regal" never
sees the bookcase until the compound is split. Both cases need explicit handling
before the retrieval model sees the tokens.

**Learning goals:**
- See why raw bi-gram frequency fails, with stop-word pairs dominating the list
- Read the PMI formula and watch its rare-word bias put chance pairs on top
- Read the likelihood-ratio formula and watch it reward well-attested phrases
- Compare PMI and LHR on the same corpus and learn which to reach for
- Split German compounds with a dictionary filter and a log-frequency score

**Prerequisites:** Sections 3.1 (tokenization) and 3.2 (stemming)

In [1]:
import re
from collections import Counter
import nltk
from nltk.collocations import BigramCollocationFinder, BigramAssocMeasures
from wordfreq import zipf_frequency
from shared.data import load_gutenberg_book
from shared.text import stopwords_for
from shared.display import print_table, display_md

In [2]:
# One-time data download (quiet, safe to re-run)
for pkg in ("stopwords", "punkt", "punkt_tab"):
    nltk.download(pkg, quiet=True)

# A Study in Scarlet (Arthur Conan Doyle), cached from Project Gutenberg
book = load_gutenberg_book(244)

# Lowercase and reduce to alphabetic-only words, as in the previous section
tokens = [t.lower() for t in nltk.word_tokenize(book.page_content) if t.isalpha()]
N = len(tokens)
STOPS = stopwords_for("en")

unigram_counts = Counter(tokens)
bigram_counts = Counter(zip(tokens, tokens[1:]))

display_md(
    f"**Corpus:** *{book.metadata['title']}* by {book.metadata['author']}\n\n"
    f"| Statistic | Value |\n|---|---|\n"
    f"| Tokens (alphabetic, lowercased) | {N:,} |\n"
    f"| Unique words | {len(unigram_counts):,} |\n"
    f"| Distinct bi-grams | {len(bigram_counts):,} |\n"
    f"| English stop words (filter) | {len(STOPS)} |"
)

**Corpus:** *A Study in Scarlet* by Doyle, Arthur Conan

| Statistic | Value |
|---|---|
| Tokens (alphabetic, lowercased) | 43,190 |
| Unique words | 5,501 |
| Distinct bi-grams | 25,946 |
| English stop words (filter) | 198 |

## 1. Why a bag of words loses phrases

The naive bag-of-words model treats a document as a multiset of independent tokens,
so "New York City" becomes three unrelated tokens. Searching for "New York" as two
terms still retrieves every document that mentions the city, so recall is fine. The
problem is precision: the query also matches "a new book from York University", and a
plain index cannot tell that from the genuine "New York" documents. Indexing the pair
as a single token `new_york` restores the signal. We keep the single tokens too, and
we emit every overlapping n-gram rather than only the longest, so that a query for
"York" or for the longer "New York City" still matches.

In [3]:
def apply_phrases(tokens, bigrams):
    """Emit each token, plus a phrase token whenever the next token forms an accepted pair."""
    result = []
    for i, tok in enumerate(tokens):
        result.append(tok)
        if i + 1 < len(tokens) and (tok, tokens[i + 1]) in bigrams:
            result.append(f"{tok}_{tokens[i + 1]}")
    return result

accepted = {("new", "york"), ("york", "city")}
example = "the city of new york city".split()

display_md(
    f"**Accepted phrase bi-grams:** `{sorted(accepted)}`\n\n"
    f"**Input tokens:** `{example}`\n\n"
    f"**After phrase injection:** `{apply_phrases(example, accepted)}`"
)

**Accepted phrase bi-grams:** `[('new', 'york'), ('york', 'city')]`

**Input tokens:** `['the', 'city', 'of', 'new', 'york', 'city']`

**After phrase injection:** `['the', 'city', 'of', 'new', 'new_york', 'york', 'york_city', 'city']`

The two overlapping phrases `new_york` and `york_city` are both emitted, next to the
single tokens. A query for "Holmes" still matches a document that only writes
"Sherlock Holmes", and a phrase query for "New York" now hits `new_york` directly
without any proximity constraint.

**Key insight:** the open question is which bi-grams are worth indexing. There are
too many to keep them all, and most ("of the", "and to") are noise. We need a scoring
function that selects the few pairs that mean something the individual tokens do not.

## 2. Naive frequency and the stop-word problem

The obvious approach is to rank bi-grams by how often they occur and keep the top of
the list. On any English text this immediately breaks.

In [4]:
print_table(
    [[f"{a} {b}", c] for (a, b), c in bigram_counts.most_common(8)],
    headers=["Bi-gram", "Frequency"],
)

| Bi-gram   |   Frequency |
|:----------|------------:|
| of the    |         303 |
| in the    |         211 |
| to the    |         136 |
| to be     |          97 |
| it was    |          95 |
| he had    |          94 |
| and the   |          90 |
| upon the  |          88 |

The head of the list is entirely pairs of stop words. Removing stop words and very
short tokens fixes the head, so we look at a longer slice to see what remains.

In [5]:
filtered = Counter(
    (a, b) for a, b in zip(tokens, tokens[1:])
    if a not in STOPS and b not in STOPS and len(a) >= 3 and len(b) >= 3
)
print_table(
    [[rank, f"{a} {b}", c] for rank, ((a, b), c) in enumerate(filtered.most_common(15), 1)],
    headers=["Rank", "Bi-gram (stop words and short tokens removed)", "Frequency"],
)

|   Rank | Bi-gram (stop words and short tokens removed)   |   Frequency |
|-------:|:------------------------------------------------|------------:|
|      1 | sherlock holmes                                 |          52 |
|      2 | jefferson hope                                  |          34 |
|      3 | john ferrier                                    |          29 |
|      4 | brixton road                                    |          13 |
|      5 | said holmes                                     |          12 |
|      6 | lucy ferrier                                    |          10 |
|      7 | salt lake                                       |          10 |
|      8 | could see                                       |           9 |
|      9 | enoch drebber                                   |           9 |
|     10 | young man                                       |           9 |
|     11 | lake city                                       |           9 |
|     12 | little girl                                     |           8 |
|     13 | said sherlock                                   |           7 |
|     14 | joseph stangerson                               |           7 |
|     15 | lauriston gardens                               |           6 |

Filtering helps: the genuinely frequent phrases now surface, and the top of the list
is real character and place names ("sherlock holmes", "jefferson hope", "brixton
road"). But frequency still cannot separate a phrase from a frequent co-occurrence.
Non-phrases ride along near the top ("said holmes", "could see", "young man", "said
sherlock") only because "said", "could", and "young" are common words, not because the
pairs mean anything together. And frequency can only ever surface phrases whose parts
are individually frequent: a rare but exclusive phrase like "lauriston gardens" barely
makes the list, and thousands more never appear at all. We need a score that measures
how strongly two words prefer each other, not just how often they occur. Two such
measures follow: pointwise mutual information (PMI) and the likelihood ratio (LHR).

## 3. Pointwise mutual information

Two words that mean something together should co-occur more often than their
individual frequencies predict under independence. PMI measures exactly that gap.

**Key formula: Pointwise Mutual Information**

$$\text{pmi}(t_1, t_2) = \log_2 \frac{p(t_1, t_2)}{p(t_1)\,p(t_2)} = \log_2 \frac{N \cdot \text{tf}(t_1, t_2)}{\text{tf}(t_1)\,\text{tf}(t_2)}$$

Here $\text{tf}(t)$ counts occurrences of a term and $\text{tf}(t_1, t_2)$ counts how
often the bi-gram occurs. PMI is the log-ratio of the observed joint probability to
the one independence would predict; it is high when two tokens almost always appear
together and rarely apart.

In [6]:
rows = []
for a, b in [("said", "holmes"), ("sherlock", "holmes")]:
    expected = unigram_counts[a] * unigram_counts[b] / N
    observed = bigram_counts[(a, b)]
    rows.append([f"{a} {b}", unigram_counts[a], unigram_counts[b],
                 observed, f"{expected:.2f}", f"{observed / expected:.0f}x"])
print_table(rows, headers=["Bi-gram", "tf(t1)", "tf(t2)", "Observed", "Expected", "Obs / Exp"])

| Bi-gram         |   tf(t1) |   tf(t2) |   Observed |   Expected | Obs / Exp   |
|:----------------|---------:|---------:|-----------:|-----------:|:------------|
| said holmes     |      207 |       98 |         12 |       0.47 | 26x         |
| sherlock holmes |       52 |       98 |         52 |       0.12 | 441x        |

"said" precedes dozens of names, so "said Holmes" beats chance only modestly. Every
occurrence of "Sherlock" is followed by "Holmes", so the pair beats chance by a factor
of several hundred. That is the association PMI is built to find. But PMI has a
weakness: it is largest when all three counts are tiny. A word that occurs once, next
to another word that occurs once, scores the maximum possible value of $\log_2 N$.

In [7]:
def build_finder(min_freq=1):
    """Bi-gram finder with a stop-word and short-token filter, optionally a frequency floor."""
    finder = BigramCollocationFinder.from_words(tokens)
    if min_freq > 1:
        finder.apply_freq_filter(min_freq)
    finder.apply_word_filter(lambda w: len(w) < 3 or w in STOPS)
    return finder

measures = BigramAssocMeasures()

finder_nofloor = build_finder(min_freq=1)
pmi_nofloor = finder_nofloor.score_ngrams(measures.pmi)
max_pmi = pmi_nofloor[0][1]
n_at_max = sum(1 for _, s in pmi_nofloor if abs(s - max_pmi) < 1e-6)
sh_rank = next(i for i, ((a, b), _) in enumerate(pmi_nofloor, 1)
               if (a, b) == ("sherlock", "holmes"))

display_md(
    f"**PMI without a frequency floor** (stop-word and short-token filter only):\n\n"
    f"- Maximum score is $\\log_2 N \\approx$ {max_pmi:.2f}\n"
    f"- {n_at_max} different bi-grams are tied at that maximum, "
    f"for example `{[f'{a} {b}' for (a, b), _ in pmi_nofloor[:3]]}`\n"
    f"- Each is a pair of words that occur once or twice and only next to each other\n"
    f"- `sherlock holmes`, the defining phrase of the book, ranks only {sh_rank}th\n\n"
    "A single chance adjacency is no evidence of a real collocation. The cure is a "
    "minimum-frequency floor: require a bi-gram to occur at least three times before "
    "scoring it, which drops the low-evidence rare pairs."
)

**PMI without a frequency floor** (stop-word and short-token filter only):

- Maximum score is $\log_2 N \approx$ 15.40
- 199 different bi-grams are tied at that maximum, for example `['admired treated', 'airy cheerfully', 'ambitious title']`
- Each is a pair of words that occur once or twice and only next to each other
- `sherlock holmes`, the defining phrase of the book, ranks only 3152th

A single chance adjacency is no evidence of a real collocation. The cure is a minimum-frequency floor: require a bi-gram to occur at least three times before scoring it, which drops the low-evidence rare pairs.

With a frequency floor of 3 and the stop-word filter, the ranked list becomes usable,
though PMI's preference for rarity still shows.

In [8]:
def score_table(finder, scored, k=8, extra_rank=None):
    """Render scored bi-grams with their raw counts. Optionally append one extra ranked row."""
    rows = []
    indices = list(range(k))
    if extra_rank is not None:
        rows_seen = {i for i in indices}
        for i, ((a, b), _) in enumerate(scored):
            if (a, b) == extra_rank and i not in rows_seen:
                indices.append(i)
                break
    for i in indices:
        (a, b), score = scored[i]
        rows.append([i + 1, f"{a} {b}", finder.ngram_fd[(a, b)],
                     finder.word_fd[a], finder.word_fd[b], f"{score:.2f}"])
    return rows

finder = build_finder(min_freq=3)
pmi_scored = finder.score_ngrams(measures.pmi)

print_table(
    score_table(finder, pmi_scored, k=6, extra_rank=("sherlock", "holmes")),
    headers=["Rank", "Bi-gram", "tf(t1,t2)", "tf(t1)", "tf(t2)", "PMI"],
)

|   Rank | Bi-gram           |   tf(t1,t2) |   tf(t1) |   tf(t2) |   PMI |
|-------:|:------------------|------------:|---------:|---------:|------:|
|      1 | audley court      |           3 |        3 |        4 | 13.4  |
|      2 | torquay terrace   |           3 |        3 |        4 | 13.4  |
|      3 | avenging angels   |           4 |        4 |        4 | 13.4  |
|      4 | lauriston gardens |           6 |        6 |        6 | 12.81 |
|      5 | sierra blanco     |           4 |        6 |        4 | 12.81 |
|      6 | secret societies  |           3 |        7 |        3 | 12.59 |
|     45 | sherlock holmes   |          52 |       52 |       98 |  8.78 |

The top rows are still rare pairs whose parts appear almost only together
("Lauriston Gardens" is the address of the murder). "Sherlock Holmes" is held far down
the list because "Holmes" is common, so the large denominator drags its PMI down. The
frequency floor removed the noise but did not fix the bias toward rarity.

## 4. Likelihood ratio

The likelihood-ratio test attacks the same problem from a hypothesis-testing angle. It
asks how much better the "these words are dependent" hypothesis explains the data than
the "these words are independent" hypothesis.

**Key formula: Log Likelihood Ratio**

$$\log \lambda = \log \frac{L(H_1)}{L(H_2)}$$

$L(H_1)$ is the likelihood under independence, $L(H_2)$ under dependence. Bi-grams are
ranked by $-2 \log \lambda$, which grows with the evidence: large when the data
strongly rules out independence. Unlike PMI, this rewards frequency, so it does not
suffer the rare-word bias. Its own weakness is the opposite one: frequent grammatical
pairs really are non-independent, so without a stop-word filter they crowd the top.

In [9]:
finder_nostop = BigramCollocationFinder.from_words(tokens)
finder_nostop.apply_freq_filter(3)
lhr_nostop = finder_nostop.score_ngrams(measures.likelihood_ratio)

print_table(
    [[i + 1, f"{a} {b}", f"{s:.0f}"] for i, ((a, b), s) in enumerate(lhr_nostop[:8])],
    headers=["Rank", "Bi-gram (no stop-word filter)", "-2 log lambda"],
)

|   Rank | Bi-gram (no stop-word filter)   |   -2 log lambda |
|-------:|:--------------------------------|----------------:|
|      1 | sherlock holmes                 |             668 |
|      2 | don t                           |             504 |
|      3 | of the                          |             489 |
|      4 | jefferson hope                  |             458 |
|      5 | to be                           |             400 |
|      6 | in the                          |             397 |
|      7 | had been                        |             377 |
|      8 | john ferrier                    |             352 |

Genuine names such as "Sherlock Holmes" and "Jefferson Hope" are interleaved with pure
grammar: "of the", "to be", "in the", "had been". The stop-word filter removes that
whole class in one step.

In [10]:
lhr_scored = finder.score_ngrams(measures.likelihood_ratio)

print_table(
    [[i + 1, f"{a} {b}", finder.ngram_fd[(a, b)],
      finder.word_fd[a], finder.word_fd[b], f"{s:.0f}"]
     for i, ((a, b), s) in enumerate(lhr_scored[:8])],
    headers=["Rank", "Bi-gram", "tf(t1,t2)", "tf(t1)", "tf(t2)", "-2 log lambda"],
)

|   Rank | Bi-gram           |   tf(t1,t2) |   tf(t1) |   tf(t2) |   -2 log lambda |
|-------:|:------------------|------------:|---------:|---------:|----------------:|
|      1 | sherlock holmes   |          52 |       52 |       98 |             668 |
|      2 | jefferson hope    |          34 |       37 |       56 |             458 |
|      3 | john ferrier      |          29 |       39 |       62 |             352 |
|      4 | brixton road      |          13 |       15 |       27 |             188 |
|      5 | salt lake         |          10 |       11 |       10 |             181 |
|      6 | lake city         |           9 |       10 |       23 |             133 |
|      7 | enoch drebber     |           9 |        9 |       62 |             119 |
|      8 | lauriston gardens |           6 |        6 |        6 |             119 |

With the filter in place, LHR puts the main characters and places of the novel on top:
Sherlock Holmes, Jefferson Hope, John Ferrier, Brixton Road. These are exactly the
phrases a reader would want to search for.

## 5. PMI or LHR: which to use

The two measures rank the same corpus differently because they measure different
things. PMI measures the *strength* of association, regardless of how often it
happens. LHR measures the *evidence* for association, which grows with frequency.
"Sherlock Holmes" makes the contrast concrete.

In [11]:
def rank_of(scored, pair):
    return next(i for i, (bg, _) in enumerate(scored, 1) if bg == pair)

pair = ("sherlock", "holmes")
display_md(
    f"**Rank of `sherlock holmes`** (frequency floor 3, stop-word filter):\n\n"
    f"| Measure | Rank | What it rewards |\n|---|---|---|\n"
    f"| PMI | {rank_of(pmi_scored, pair)} of {len(pmi_scored)} | rare, exclusive pairs |\n"
    f"| LHR | {rank_of(lhr_scored, pair)} of {len(lhr_scored)} | well-attested pairs |\n\n"
    "The same phrase is buried under PMI and first under LHR. Because the two optimise "
    "different quantities, a genuine phrase can score well on one and poorly on the other."
)

**Rank of `sherlock holmes`** (frequency floor 3, stop-word filter):

| Measure | Rank | What it rewards |
|---|---|---|
| PMI | 45 of 115 | rare, exclusive pairs |
| LHR | 1 of 115 | well-attested pairs |

The same phrase is buried under PMI and first under LHR. Because the two optimise different quantities, a genuine phrase can score well on one and poorly on the other.

Lining up the two top lists side by side shows the split cleanly.

In [12]:
k = 10
pmi_top = [f"{a} {b}" for (a, b), _ in pmi_scored[:k]]
lhr_top = [f"{a} {b}" for (a, b), _ in lhr_scored[:k]]
print_table(
    [[i + 1, pmi_top[i], lhr_top[i]] for i in range(k)],
    headers=["Rank", "Top by PMI", "Top by LHR"],
)

|   Rank | Top by PMI          | Top by LHR        |
|-------:|:--------------------|:------------------|
|      1 | audley court        | sherlock holmes   |
|      2 | torquay terrace     | jefferson hope    |
|      3 | avenging angels     | john ferrier      |
|      4 | lauriston gardens   | brixton road      |
|      5 | sierra blanco       | salt lake         |
|      6 | secret societies    | lake city         |
|      7 | chemical laboratory | enoch drebber     |
|      8 | salt lake           | lauriston gardens |
|      9 | madame charpentier  | scotland yard     |
|     10 | smouldering fire    | lucy ferrier      |

Three cases explain the difference:

- **Rare but tight** phrases occur a handful of times, always as a unit. They top
  PMI and sink under LHR: named places and technical terms live here.
- **Frequent but loose** pairs such as "young man" occur often but are not phrases.
  LHR scores them moderately, PMI correctly discounts them.
- **Frequent and tight** phrases such as "Sherlock Holmes" score high on both and
  need no adjudication.

For a phrase vocabulary that maximises recall, take the top-$k$ from both and union
them: LHR contributes the well-attested head, PMI the rare-but-specific tail.

In [13]:
pmi_set = {bg for bg, _ in pmi_scored[:20]}
lhr_set = {bg for bg, _ in lhr_scored[:20]}
only_pmi = pmi_set - lhr_set
only_lhr = lhr_set - pmi_set

display_md(
    f"**Union of the top 20 from each measure:**\n\n"
    f"- In both lists: {len(pmi_set & lhr_set)} bi-grams\n"
    f"- Contributed only by PMI (rare, specific): "
    f"`{[f'{a} {b}' for a, b in list(only_pmi)[:5]]}`\n"
    f"- Contributed only by LHR (frequent, attested): "
    f"`{[f'{a} {b}' for a, b in list(only_lhr)[:5]]}`\n\n"
    "The union only adds candidates, so plan a downstream prune to drop the "
    "frequent-but-loose pairs LHR lets through. The filters matter more than the "
    "choice of measure: a frequency floor removes PMI's chance pairs, a stop-word "
    "filter removes LHR's grammatical pairs, and without both neither is usable."
)

**Union of the top 20 from each measure:**

- In both lists: 7 bi-grams
- Contributed only by PMI (rare, specific): `['smouldering fire', 'nearly five', 'secret societies', 'alkali plain', 'twenty years']`
- Contributed only by LHR (frequent, attested): `['john ferrier', 'lucy ferrier', 'sherlock holmes', 'young hunter', 'chapter iii']`

The union only adds candidates, so plan a downstream prune to drop the frequent-but-loose pairs LHR lets through. The filters matter more than the choice of measure: a frequency floor removes PMI's chance pairs, a stop-word filter removes LHR's grammatical pairs, and without both neither is usable.

## 6. Splitting German compounds

A phrase glues tokens together; a compound hides several words inside one token.
German makes this productive: "Wolkenkratzer" (skyscraper) combines "Wolke" (cloud)
and "Kratzer" (scratcher). A user who types "Etikettierung Gesetz" cannot match a
document indexed only under the full compound. The fix is to index both the compound
and its parts.

**Caution: not every compound splits usefully.** Endocentric compounds derive their
meaning from their parts ("Abfalleimer" is a kind of Eimer, a bucket), so splitting is
safe. Exocentric compounds do not: a "Wolkenkratzer" is not a kind of "Kratzer"
(scratcher), so splitting it adds tokens with the wrong sense and can hurt precision.
Whether the recall gain is worth it depends on the collection.

Splitting proceeds in two stages: generate candidate splits, then score them. For
scoring we use a German word-frequency table (the `wordfreq` package, offline), which
returns a Zipf value: higher means more common, and 0 means the word never appears.

**Key formula: Compound Split Score**

$$\text{score}(S) = \frac{1}{|S|} \sum_{p_i \in S} \log \frac{\text{tf}(p_i)}{N}$$

The average log-frequency of the parts. A higher score means the split is into more
common words, so a more plausible decomposition. The Zipf value is this log-frequency
on a fixed scale, so averaging Zipf values ranks splits the same way. We treat a part
as a dictionary word only when its Zipf value clears a small threshold, which both
enforces the dictionary filter and rejects spurious short fragments.

In [14]:
MIN_ZIPF = 3.5                        # a part must be at least this common to count as a word

def zipf(word):
    return zipf_frequency(word.lower(), "de")

def is_word(word):
    return zipf(word) >= MIN_ZIPF     # a real, reasonably common German word

def avg_score(parts):
    return sum(zipf(p) for p in parts) / len(parts)

First reproduce the worked example from the book. We score three candidate splits of
"Autobahnausfahrt" (motorway exit), discarding any split with a part that never
appears in the reference corpus.

In [15]:
candidates = [
    ["Auto", "Bahn", "Ausfahrt"],
    ["Autobahn", "Ausfahrt"],
    ["Auto", "Bahnausfahrt"],
]
rows = []
for parts in candidates:
    zipfs = [zipf(p) for p in parts]
    if all(is_word(p) for p in parts):
        verdict = f"keep (avg {avg_score(parts):.2f})"
    else:
        verdict = "discarded (part not in dictionary)"
    rows.append([" + ".join(parts), ", ".join(f"{z:.1f}" for z in zipfs), verdict])
print_table(rows, headers=["Split", "Parts (Zipf)", "Verdict"])

| Split                  | Parts (Zipf)   | Verdict                            |
|:-----------------------|:---------------|:-----------------------------------|
| Auto + Bahn + Ausfahrt | 5.3, 5.0, 3.7  | keep (avg 4.67)                    |
| Autobahn + Ausfahrt    | 4.4, 3.7       | keep (avg 4.05)                    |
| Auto + Bahnausfahrt    | 5.3, 0.0       | discarded (part not in dictionary) |

The dictionary filter throws out (Auto, Bahnausfahrt) because "Bahnausfahrt" never
appears. Of the two survivors, plain average frequency ranks the finer split
(Auto, Bahn, Ausfahrt) above the more natural (Autobahn, Ausfahrt), because "Auto" and
"Bahn" are extremely common words. A frequency score alone tends to over-split into
short common words. Now generate the candidates automatically instead of writing them
by hand.

In [16]:
LINKS = ["s", "es", "n", "en", "er"]      # German linking morphemes (Fugen)

def candidate_splits(word, min_len=4, max_parts=3):
    """All ways to split a word into known dictionary parts, allowing linking morphemes."""
    results = []
    def extend(rest, acc):
        if len(acc) >= max_parts:
            return
        for i in range(min_len, len(rest) + 1):
            head, remainder = rest[:i], rest[i:]
            if not is_word(head):
                continue
            if not remainder:
                if acc:                    # at least two parts overall
                    results.append(acc + [head])
            else:
                extend(remainder, acc + [head])
                for link in LINKS:
                    if remainder.startswith(link) and len(remainder) - len(link) >= min_len:
                        extend(remainder[len(link):], acc + [head])
    extend(word.lower(), [])
    unique = []
    for parts in results:
        if parts not in unique:
            unique.append(parts)
    return unique

splits = candidate_splits("Autobahnausfahrt")
print_table(
    [[" + ".join(p), f"{avg_score(p):.2f}"]
     for p in sorted(splits, key=avg_score, reverse=True)[:4]],
    headers=["Candidate split", "Avg Zipf"],
)

| Candidate split        |   Avg Zipf |
|:-----------------------|-----------:|
| auto + bahn + ausfahrt |       4.67 |
| autobahn + ausfahrt    |       4.05 |

Ranked by average alone, the finer split wins, just as in the worked example. The book
notes the standard fix: penalise splits with more or shorter parts. A small per-part
penalty is enough to prefer the natural decomposition.

In [17]:
def penalized_score(parts, beta=1.3):
    return avg_score(parts) - beta * (len(parts) - 1)

print_table(
    [[" + ".join(p), f"{avg_score(p):.2f}", f"{penalized_score(p):.2f}"]
     for p in sorted(splits, key=penalized_score, reverse=True)[:4]],
    headers=["Candidate split", "Avg Zipf", "Penalized"],
)

| Candidate split        |   Avg Zipf |   Penalized |
|:-----------------------|-----------:|------------:|
| autobahn + ausfahrt    |       4.05 |        2.75 |
| auto + bahn + ausfahrt |       4.67 |        2.07 |

The penalty flips the ranking: (Autobahn, Ausfahrt) now beats the over-split
(Auto, Bahn, Ausfahrt). We do not have to commit to a single winner, though. Favouring
recall, we keep the top few plausible splits and index the union of their parts
alongside the full compound.

In [18]:
def best_split(word):
    splits = candidate_splits(word)
    return max(splits, key=penalized_score) if splits else [word]

def index_terms(word, keep=2):
    top = sorted(candidate_splits(word), key=penalized_score, reverse=True)[:keep]
    parts = {word.lower()}
    for split in top:
        parts.update(split)
    return sorted(parts)

compounds = ["Wolkenkratzer", "Bücherregal", "Abfalleimer", "Schneemann", "Autobahnausfahrt"]
print_table(
    [[c, " + ".join(best_split(c)), ", ".join(index_terms(c))] for c in compounds],
    headers=["Compound", "Best split", "Indexed terms (union)"],
)

| Compound         | Best split          | Indexed terms (union)                            |
|:-----------------|:--------------------|:-------------------------------------------------|
| Wolkenkratzer    | wolken + kratzer    | kratzer, wolke, wolken, wolkenkratzer            |
| Bücherregal      | bücher + regal      | bücher, bücherregal, regal                       |
| Abfalleimer      | abfall + eimer      | abfall, abfalleimer, eimer                       |
| Schneemann       | schnee + mann       | mann, schnee, schneemann                         |
| Autobahnausfahrt | autobahn + ausfahrt | ausfahrt, auto, autobahn, autobahnausfahrt, bahn |

Every compound now contributes both itself and its parts to the index, so a query for
"Regal", "Eimer", or "Ausfahrt" reaches the documents that only ever wrote the full
word. "Wolkenkratzer" is the exocentric warning made concrete: it splits into "Wolken"
(clouds) and "Kratzer" (scratcher), neither of which means skyscraper, so the extra
tokens trade some precision for the recall gain.

**Note on the reference corpus:** the score depends entirely on the German frequency
table behind `wordfreq`. A rare but real word can fall below the threshold and be
discarded, and rare technical compounds may lose good splits. A production splitter
pairs a larger dictionary with a curated list of known compounds.

## Summary

| Method | What it computes | Strength | Weakness |
| --- | --- | --- | --- |
| Raw frequency | Count co-occurrences | Trivial | Stop-word pairs dominate |
| PMI | log-ratio of joint to independent probability | Finds exclusive pairs | Biased to rare pairs; needs a frequency floor |
| LHR | hypothesis test, dependent vs independent | Robust to frequency | Ranks grammatical pairs; needs a stop-word filter |
| Compound split | dictionary filter + average log-frequency | Recovers hidden parts | Over-splits into common words; needs a penalty |

<div style="border-left: 4px solid #C8102E; background: rgba(200, 16, 46, 0.06); padding: 0.6em 0.9em; margin: 0.6em 0; border-radius: 4px;">
<strong style="color:#C8102E; text-transform:uppercase; font-size:0.78em; letter-spacing:0.06em;">Takeaway</strong><br>
PMI finds the strength of an association and favours rare, exclusive pairs; LHR finds the evidence for an association and favours well-attested ones. For a recall-oriented phrase vocabulary, union the top of both and prune downstream. Compound splitting is the mirror image: generate candidate parts, keep only real words, and score plausibility by log-frequency, indexing the union of the best splits alongside the full compound. All three techniques still run in production lexical retrievers.
</div>

## Try it yourself

1. Change the frequency floor and watch the PMI and LHR lists move.
2. Split a compound of your own and inspect every candidate the generator proposes.

In [19]:
def top_bigrams(measure="lhr", min_freq=3, k=8):
    finder = build_finder(min_freq=min_freq)
    scorer = measures.likelihood_ratio if measure == "lhr" else measures.pmi
    scored = finder.score_ngrams(scorer)
    return [f"{a} {b}" for (a, b), _ in scored[:k]]

my_word = "Bundesausbildungsförderungsgesetz"
display_md(
    f"**LHR top 5, floor 3:** `{top_bigrams('lhr', min_freq=3, k=5)}`\n\n"
    f"**LHR top 5, floor 6:** `{top_bigrams('lhr', min_freq=6, k=5)}`\n\n"
    f"**PMI top 5, floor 6:** `{top_bigrams('pmi', min_freq=6, k=5)}`\n\n"
    f"**Best split of `{my_word}`:** `{' + '.join(best_split(my_word))}`"
)

**LHR top 5, floor 3:** `['sherlock holmes', 'jefferson hope', 'john ferrier', 'brixton road', 'salt lake']`

**LHR top 5, floor 6:** `['sherlock holmes', 'jefferson hope', 'john ferrier', 'brixton road', 'salt lake']`

**PMI top 5, floor 6:** `['lauriston gardens', 'salt lake', 'scotland yard', 'lake city', 'private hotel']`

**Best split of `Bundesausbildungsförderungsgesetz`:** `Bundesausbildungsförderungsgesetz`